# 🚀 EL ÁLBUM B — V10.0 MASTER
---
⚠️ **ANTES DE EMPEZAR: Activa la GPU**
Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU** → Guardar

**Uso normal:** ejecuta **Celda 1 → Celda 3**

**Uso manual:** ejecuta **Celda 1 → Celda 2**

## CELDA 1 — Instalación + Motor V10.0
Instala dependencias y carga el motor. Ejecuta **siempre primero**.

In [ ]:
import subprocess, sys

# --- Instalar dependencias ---
print('⏳ Verificando dependencias...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy>=2.3.0', '--upgrade', '-q'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'onnxruntime', 'onnxruntime-gpu', '-y', '-q'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnxruntime==1.19.2', '-q'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'rembg==2.0.76', 'huggingface_hub', 'Pillow', '-q'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers>=0.30.0', 'transformers', 'accelerate', 'ipywidgets', '-q'], check=False)
print('✅ Dependencias listas.\n')

# --- Cargar librerías ---
import os, glob
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageChops

try:
    import rembg
    _new_session = getattr(rembg, 'new_session', None)
    if _new_session is None:
        from rembg.session_factory import new_session as _new_session
    from rembg import remove
    from google.colab import drive
    import torch
    from diffusers import StableDiffusionXLPipeline
    print('✅ Librerías cargadas correctamente.')
except ModuleNotFoundError as e:
    print(f'❌ Error cargando librerías: {e}')
    raise SystemExit

_model_cache = {}

def _cargar_modelo(tema):
    modelo_id = 'Lykon/dreamshaper-xl-1-0' if tema == 'magic' else 'stabilityai/stable-diffusion-xl-base-1.0'
    for key in list(_model_cache.keys()):
        if key != modelo_id:
            del _model_cache[key]
            torch.cuda.empty_cache()
            print('   🗑️ Modelo anterior liberado.')
    if modelo_id not in _model_cache:
        print(f'   [⏳] Cargando modelo {modelo_id.split("/")[1]}...')
        pipe = StableDiffusionXLPipeline.from_pretrained(
            modelo_id,
            torch_dtype=torch.float16,
            use_safetensors=True,
            variant='fp16',
        )
        pipe.enable_model_cpu_offload()
        pipe.enable_attention_slicing(1)
        pipe.unet.to(memory_format=torch.channels_last)
        _model_cache[modelo_id] = pipe
        print('   ✅ Modelo listo.')
    return _model_cache[modelo_id]


class ElAlbumB_V10_0_Master:
    def __init__(self):
        print('\n🔗 Conectando con Google Drive...')
        drive.mount('/content/drive', force_remount=True)
        self.ruta_base = '/content/drive/MyDrive/El Álbum B - Clientes/'
        self.session = _new_session('isnet-general-use')
        print('✅ Motor V10.0 inicializado.')

        self.banco_prompts = {
            'finca':     'wide angle rolling green hills, vibrant sunset sky, golden hour, cinematic lighting, sharp focus, photorealistic, 8k, no people',
            'vogue':     'luxury modern villa terrace, coastal ocean sunset, infinity pool, golden hour, editorial photography, sharp focus, no people',
            'nostalgia': 'colonial courtyard at sunset, terracotta walls, warm golden light, romantic atmosphere, sharp focus, photorealistic, 8k, no people',
            'magic':     'aurora borealis purple green swirling sky, bioluminescent glowing flowers, floating light orbs bokeh, enchanted forest, fairy lights, fantasy digital art, by alan lee, artstation hq, no people',
            'cinematic': 'dramatic cinematic landscape, rolling hills, epic saturated twilight sky, deep rich contrast, volumetric light, professional photography, 8k, no people',
        }
        self.negative_prompts = {
            'magic':   'photo, photography, realistic, landscape, people, humans, clouds, sunset, hills, ugly, blurry',
            'default': 'people, humans, couple, text, watermark, blurry, ugly, deformed',
        }

    def _generar_fondo(self, tema, ancho, alto):
        pipe = _cargar_modelo(tema)
        negative = self.negative_prompts.get(tema, self.negative_prompts['default'])
        result = pipe(
            prompt=self.banco_prompts[tema],
            width=768, height=768,
            num_inference_steps=20,
            guidance_scale=7.5,
            negative_prompt=negative,
        ).images[0]
        return result.resize((ancho, alto), resample=Image.Resampling.LANCZOS)

    def _inyectar_micro_grano_fondo(self, img_pil, intensidad=3.0):
        arr = np.array(img_pil).astype(np.float32)
        luma = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
        modulador = np.clip(1.0 - (np.abs(luma - 120.0) / 150.0), 0.2, 1.0)
        ruido = np.random.normal(0, intensidad, luma.shape) * modulador
        for i in range(3):
            arr[..., i] += ruido
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    def _aplicar_high_pass_sharpen(self, img_pil, radio=1.5, fuerza=1.3):
        arr_orig = np.array(img_pil).astype(np.float32)
        arr_blur = np.array(img_pil.filter(ImageFilter.GaussianBlur(radius=radio))).astype(np.float32)
        return Image.fromarray(np.clip(arr_orig + fuerza * (arr_orig - arr_blur), 0, 255).astype(np.uint8))

    def _optimizar_nitidez_rostros(self, img_pil):
        fino = img_pil.filter(ImageFilter.UnsharpMask(radius=1.2, percent=140, threshold=2))
        pop = ImageEnhance.Contrast(fino).enhance(1.08)
        return ImageEnhance.Brightness(pop).enhance(1.04)

    def _aislar_region_clara_vestido(self, img_original_rgb, alpha_mask):
        arr_alpha = np.array(alpha_mask)
        arr_luma = np.array(img_original_rgb.convert('L'))
        arr_seed = ((arr_alpha > 240) & (arr_luma > 125)).astype(np.uint8) * 255
        return Image.fromarray(arr_seed).filter(ImageFilter.MaxFilter(35))

    def _descontaminar_halos_negros(self, img_rgb, alpha_mask, mascara_vestido_velo):
        arr = np.array(img_rgb).astype(np.float32)
        arr_alpha = np.array(alpha_mask).astype(np.float32) / 255.0
        arr_zona = np.array(mascara_vestido_velo)
        frontera = (arr_alpha > 0.01) & (arr_alpha < 0.96) & (arr_zona > 0)
        factor = (1.0 - arr_alpha) * 0.95
        marfil = [246.0, 244.0, 240.0]
        for i in range(3):
            reconstruido = arr[..., i] + (marfil[i] - arr[..., i]) * factor
            arr[..., i] = np.where(frontera, np.clip(reconstruido, 0, 255), arr[..., i])
        return Image.fromarray(arr.astype(np.uint8))

    def _aplicar_light_wrap(self, composicion_rgb, alpha_mask, fondo_pil, radio=4, intensidad=0.12):
        radio_impar = radio if radio % 2 != 0 else max(3, radio - 1)
        fondo_blur = fondo_pil.filter(ImageFilter.GaussianBlur(radius=radio_impar * 2))
        alpha_encogido = alpha_mask.filter(ImageFilter.MinFilter(radio_impar))
        borde_interno = ImageChops.subtract(alpha_mask, alpha_encogido)
        borde_suave = borde_interno.filter(ImageFilter.GaussianBlur(radius=radio_impar / 2))
        if intensidad < 1.0:
            borde_suave = ImageEnhance.Brightness(borde_suave).enhance(intensidad)
        resultado = composicion_rgb.copy()
        resultado.paste(fondo_blur, (0, 0), borde_suave)
        return resultado

    def revelar_foto_master(self, ruta_entrada, ruta_salida, tema):
        try:
            img_original = Image.open(ruta_entrada).convert('RGB')
            ancho_orig, alto_orig = img_original.size

            # Limitar lado largo a 4000px
            MAX_LADO = 4000
            if max(ancho_orig, alto_orig) > MAX_LADO:
                factor = MAX_LADO / max(ancho_orig, alto_orig)
                img_original = img_original.resize(
                    (int(ancho_orig * factor), int(alto_orig * factor)),
                    resample=Image.Resampling.LANCZOS
                )
                ancho_orig, alto_orig = img_original.size
                print(f'   ⚠️ Foto redimensionada a {ancho_orig}x{alto_orig} px')

            print('   [⏳] [1] Alpha Matting de alta precisión...')
            img_rgba = remove(
                img_original, session=self.session,
                alpha_matting=True,
                alpha_matting_erode_size=2,
                alpha_matting_foreground_threshold=235,
                alpha_matting_background_threshold=20,
            )
            alpha_nativo = img_rgba.split()[3]
            alpha_rasurada = alpha_nativo.filter(ImageFilter.MinFilter(3))
            alpha_suavizado = alpha_rasurada.filter(ImageFilter.GaussianBlur(radius=1.2))
            alpha_final = ImageChops.multiply(alpha_suavizado, alpha_suavizado)
            arr_alpha = np.array(alpha_final)
            h, w = arr_alpha.shape
            bottom = arr_alpha[int(h * 0.85):, :]
            bottom[bottom < 80] = 0
            arr_alpha[int(h * 0.85):, :] = bottom
            alpha_final = Image.fromarray(arr_alpha)
            sujetos_rgb = img_rgba.convert('RGB')

            print(f'   [⏳] [6] Generando fondo ({tema})...')
            fondo_ia = self._generar_fondo(tema, ancho_orig, alto_orig)
            fondo_texturizado = self._inyectar_micro_grano_fondo(fondo_ia, intensidad=2.8)

            print('   [⏳] [5] Discriminando áreas textiles para anti-halos...')
            mascara_vestido = self._aislar_region_clara_vestido(img_original, alpha_final)
            sujetos_descontaminados = self._descontaminar_halos_negros(sujetos_rgb, alpha_final, mascara_vestido)

            print('   [⏳] [2] Optimizando nitidez en rostros...')
            sujetos_enfocados = self._optimizar_nitidez_rostros(sujetos_descontaminados)
            print('   [⏳] [3] High Pass micro-detalle estructural...')
            sujetos_high_pass = self._aplicar_high_pass_sharpen(sujetos_enfocados)
            sujetos_vividos = ImageEnhance.Color(sujetos_high_pass).enhance(1.15)
            sujetos_finales = ImageEnhance.Contrast(sujetos_vividos).enhance(1.06)

            composicion = fondo_texturizado.copy()
            composicion.paste(sujetos_finales, (0, 0), alpha_final)

            print('   [⏳] [4] Light Wrap adaptativo...')
            composicion_con_wrap = self._aplicar_light_wrap(composicion, alpha_final, fondo_texturizado)
            img_entregable = ImageEnhance.Contrast(composicion_con_wrap).enhance(1.01)

            icc = img_original.info.get('icc_profile')
            save_kwargs = {'format': 'JPEG', 'quality': 100, 'subsampling': 0}
            if icc:
                save_kwargs['icc_profile'] = icc
            img_entregable.save(ruta_salida, **save_kwargs)
            return True

        except Exception as e:
            import traceback
            print(f'   ❌ Error: {e}')
            traceback.print_exc()
            return False

    def procesar_lote_master(self, email_cliente, tema):
        import shutil
        carpeta_cliente    = os.path.join(self.ruta_base, email_cliente.strip().lower())
        carpeta_upscale    = os.path.join(carpeta_cliente, 'pre_upscale')
        carpeta_resultados = os.path.join(carpeta_cliente, 'resultados_finales_x2')

        if not os.path.exists(carpeta_cliente):
            print(f'❌ Carpeta no encontrada: {carpeta_cliente}')
            return

        if os.path.exists(carpeta_upscale):
            shutil.rmtree(carpeta_upscale)
        os.makedirs(carpeta_upscale)

        archivos = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            archivos.extend(glob.glob(os.path.join(carpeta_cliente, ext)))

        excluir = ['_v', '_WRAP_', '_restored', 'pre_upscale']
        imagenes_validas = [f for f in archivos if not any(v in f for v in excluir)]
        total = len(imagenes_validas)

        if total == 0:
            print('⚠️ No hay archivos originales válidos en la raíz del cliente.')
            return

        print(f'\n⚡ MOTOR V10.0 — {total} foto(s) · tema: {tema} ⚡\n')
        exitosos = 0

        for indice, ruta_original in enumerate(imagenes_validas, start=1):
            nombre = os.path.basename(ruta_original)
            nombre_base, _ = os.path.splitext(nombre)
            ruta_salida = os.path.join(carpeta_upscale, f'{nombre_base}_{tema}_v100.jpg')
            print(f'⏳ [{indice}/{total}] Procesando: {nombre}')
            if self.revelar_foto_master(ruta_original, ruta_salida, tema):
                print(f'   ✅ Guardado en: pre_upscale/')
                exitosos += 1
            else:
                print(f'   ⚠️ Omitido por error.')

        print(f'\n📊 Resultado: {exitosos}/{total} fotos procesadas correctamente.')
        self._carpeta_upscale    = carpeta_upscale
        self._carpeta_resultados = carpeta_resultados


def _ejecutar_realesrgan(carpeta_input, carpeta_output):
    os.makedirs(carpeta_output, exist_ok=True)
    if not os.path.exists('/content/Real-ESRGAN'):
        os.system('git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN -q')
        print('✅ Real-ESRGAN clonado.')
    os.chdir('/content/Real-ESRGAN')
    if not os.path.exists('/content/Real-ESRGAN/realesrgan.egg-info'):
        print('Instalando dependencias Real-ESRGAN...')
        os.system('pip install "basicsr @ git+https://github.com/XPixelGroup/BasicSR.git" -q')
        os.system('pip install facexlib gfpgan -q')
        os.system('pip install -e . -q')
        print('✅ Instalado.')
    if not os.path.exists('/content/Real-ESRGAN/weights/RealESRGAN_x4plus.pth'):
        os.system('wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights/ -q')
        print('✅ Pesos descargados.')
    print('🚀 Iniciando Real-ESRGAN x2...')
    os.system(f'python inference_realesrgan.py -n RealESRGAN_x4plus -i "{carpeta_input}" -o "{carpeta_output}" --outscale 2 --tile 400')
    fotos = glob.glob(os.path.join(carpeta_output, '*'))
    print(f'✅ {len(fotos)} foto(s) guardadas en: {carpeta_output}')


print('\n✅ Motor V10.0 listo.')
print('   → Modo automático: ejecuta la Celda 3')
print('   → Modo manual:     ejecuta la Celda 2')

## CELDA 2 — Procesado manual
Usa esta celda si quieres procesar un cliente concreto manualmente.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

email_input = widgets.Text(
    value='cliente@email.com',
    description='📩 Cliente:',
    layout=widgets.Layout(width='350px')
)

temas = {
    '🌾 Finca Cristalina':  'finca',
    '🌟 Vogue Chic':        'vogue',
    '🎞️ Nostalgia Film':    'nostalgia',
    '🌌 Magic Wonderland':  'magic',
    '🎬 Cinematic Drama':   'cinematic',
}

botones = widgets.ToggleButtons(
    options=list(temas.keys()),
    description='🌅 Tema:',
    button_style='info',
    layout=widgets.Layout(width='100%')
)

boton_iniciar = widgets.Button(
    description='⚡ INICIAR PROCESADO',
    button_style='success',
    layout=widgets.Layout(width='300px', height='45px')
)

output = widgets.Output()

def iniciar(b):
    with output:
        clear_output()
        email = email_input.value.strip()
        tema  = temas[botones.value]
        print(f'📩 Cliente: {email}')
        print(f'🌅 Tema: {botones.value}')
        print('─' * 50)
        global motor
        motor = ElAlbumB_V10_0_Master()
        motor.procesar_lote_master(email, tema)
        _ejecutar_realesrgan(motor._carpeta_upscale, motor._carpeta_resultados)
        print('\n🎉 ¡Listo! Revisa la Celda 4 para ver los resultados.')

boton_iniciar.on_click(iniciar)
display(widgets.VBox([email_input, botones, boton_iniciar, output]))

## CELDA 3 — Monitor automático 🤖
Procesa automáticamente cada pedido que llegue del formulario.

**Pulsa STOP para detener.**

In [ ]:
import json, time, os, glob

RUTA_RAIZ = '/content/drive/MyDrive/El Álbum B - Clientes/'
PENDING   = os.path.join(RUTA_RAIZ, 'pending.json')
INTERVALO = 60

motor = ElAlbumB_V10_0_Master()

print('🤖 Monitor automático activo.')
print(f'   Comprobando cada {INTERVALO} segundos...')
print('   Pulsa STOP para detener.\n')

while True:
    try:
        if os.path.exists(PENDING):
            with open(PENDING, 'r') as f:
                cola = json.load(f)
            if not isinstance(cola, list):
                cola = [cola]
            if cola:
                trabajo = cola.pop(0)
                email   = trabajo.get('email', '')
                tema    = trabajo.get('tema', 'cinematic')
                with open(PENDING, 'w') as f:
                    json.dump(cola, f)
                print(f'\n📩 [{time.strftime("%H:%M:%S")}] Nuevo trabajo: {email} | tema: {tema}')
                print('─' * 60)
                motor.procesar_lote_master(email, tema)
                _ejecutar_realesrgan(motor._carpeta_upscale, motor._carpeta_resultados)
                print(f'\n🎉 [{time.strftime("%H:%M:%S")}] Completado: {email}')
                print('─' * 60)
            else:
                print(f'[{time.strftime("%H:%M:%S")}] Sin pedidos. Esperando...', end='\r')
        else:
            print(f'[{time.strftime("%H:%M:%S")}] Sin pedidos. Esperando...', end='\r')
    except Exception as e:
        print(f'\n⚠️ Error: {e}. Reintentando en {INTERVALO}s...')
    time.sleep(INTERVALO)

## CELDA 4 — Vista previa de resultados

In [ ]:
import glob, os
from PIL import Image
import matplotlib.pyplot as plt

try:
    carpeta_ver = motor._carpeta_resultados
except NameError:
    EMAIL_CLIENTE = 'cliente@email.com'
    carpeta_ver = f'/content/drive/MyDrive/El Álbum B - Clientes/{EMAIL_CLIENTE}/resultados_finales_x2'

fotos = glob.glob(os.path.join(carpeta_ver, '*.jpg')) + glob.glob(os.path.join(carpeta_ver, '*.png'))

if not fotos:
    print('⚠️ No se encontraron fotos en:', carpeta_ver)
else:
    print(f'📸 {len(fotos)} foto(s) en resultados_finales_x2:')
    for ruta in fotos[:6]:
        img = Image.open(ruta)
        thumb = img.copy()
        thumb.thumbnail((700, 700))
        plt.figure(figsize=(9, 6))
        plt.imshow(thumb)
        plt.title(os.path.basename(ruta), fontsize=10)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print(f'  ✅ {os.path.basename(ruta)} — {img.size[0]}x{img.size[1]} px')